# E3D Tutorial — Beginner / Portable Version

This version is intended as a first tutorial for students.

**Goal:** run an existing E3D/Allegro-FM model on one small molecule and inspect the pairwise E3D values \(D_{ij}\).

You do **not** need to understand every Python line at first. Run the cells from top to bottom and focus on:

1. what structure is given to the model,
2. what atomic pairs are found,
3. what \(D_{ij}\) values are obtained.

This notebook is based on `E3D_tutorials_v2.ipynb`, but the force-debugging section and the Al₂O₃–SiC interface tutorial have been removed for the first exercise.


## 0. Environment setup (Google Colab / local Jupyter)

This notebook is designed to run in either **Google Colab** or a **local Jupyter environment**.

- On the first run, missing Python packages are installed into the **same Python environment used by this notebook kernel**.
- On later runs, already-installed packages are reused.
- The E3D/Allegro-FM checkpoint is downloaded only if it is not already present in the current working directory.
- CPU is sufficient for this small tutorial. CUDA is used automatically when available.

> **Local PC note:** Current NequIP/Allegro releases require Python 3.10 or newer.  
> If package installation fails because of permissions or environment conflicts, use a fresh virtual environment and start Jupyter from that environment.


In [2]:
# Portable setup: works in Google Colab and local Jupyter
import sys
import subprocess
import importlib.util
import importlib.metadata
import platform
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} is being used. "
        "Please use Python 3.10 or newer for current NequIP/Allegro."
    )

# 安全な Colab 判定
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print("Environment :", "Google Colab" if IS_COLAB else "Local Jupyter")
print("Python      :", sys.version.split()[0])
print("Executable  :", sys.executable)
print("OS          :", platform.platform())

# Set False if you do not want this notebook to install missing packages automatically.
AUTO_INSTALL = True

def ensure_package(import_name, pip_name=None):
    """Install a package only when its import module is missing."""
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        if not AUTO_INSTALL:
            raise ImportError(
                f"Missing package: {import_name}. "
                f"Install it with: {sys.executable} -m pip install {pip_name}"
            )
        print(f"\nInstalling {pip_name} ...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", pip_name]
        )
    else:
        print(f"OK: {import_name}")

ensure_package("torch", "torch")

# Allegro is distributed on PyPI as nequip-allegro.
if importlib.util.find_spec("nequip") is None or importlib.util.find_spec("allegro") is None:
    if not AUTO_INSTALL:
        raise ImportError(
            "NequIP/Allegro is missing. Install with: "
            f"{sys.executable} -m pip install nequip-allegro"
        )
    print("\nInstalling nequip-allegro ...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "nequip-allegro"]
    )
else:
    print("OK: nequip / allegro")

ensure_package("numpy", "numpy")
ensure_package("ase", "ase")
ensure_package("py3Dmol", "py3Dmol")
ensure_package("gdown", "gdown")

# Import after installation
import numpy as np
import torch
import gdown
import py3Dmol
import ase

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKDIR = Path.cwd()

print("\n--- Ready ---")
print("Working dir :", WORKDIR)
print("PyTorch     :", torch.__version__)
print("Device      :", device)

try:
    print("NequIP      :", importlib.metadata.version("nequip"))
except Exception:
    pass
try:
    print("Allegro     :", importlib.metadata.version("nequip-allegro"))
except Exception:
    pass

Environment : Local Jupyter
Python      : 3.10.21
Executable  : /home/mamem/miniforge3/envs/bond-analyzer/bin/python
OS          : Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
OK: torch
OK: nequip / allegro
OK: numpy
OK: ase
OK: py3Dmol
OK: gdown

--- Ready ---
Working dir : /home/mamem/research/interatomic-bond-analyzer
PyTorch     : 2.14.1+cpu
Device      : cpu
NequIP      : 0.19.1
Allegro     : 0.8.3


## 1. Download the E3D / Allegro-FM model

The model is saved in the notebook's current working directory.  
Therefore, the same code works in Colab and on a local PC without hard-coding `/content`.


In [3]:
MODEL_NAME = "s128_t64.nequip.ckpt"
MODEL_FILE_ID = "13qNINxBcToLHlWKY5OmKsnXpsdbzzQbk"
ckpt_path = WORKDIR / MODEL_NAME

if ckpt_path.exists():
    print(f"Model already exists: {ckpt_path}")
else:
    print(f"Downloading model to: {ckpt_path}")
    result = gdown.download(
        f"https://drive.google.com/uc?id={MODEL_FILE_ID}",
        str(ckpt_path),
        quiet=False,
    )
    if result is None or not ckpt_path.exists():
        raise RuntimeError(
            "Model download failed. Check the internet connection and Google Drive access."
        )

print("Checkpoint:", ckpt_path)


Downloading...
From: https://drive.google.com/uc?id=13qNINxBcToLHlWKY5OmKsnXpsdbzzQbk
To: /home/mamem/research/interatomic-bond-analyzer/s128_t64.nequip.ckpt
100%|██████████| 7.43M/7.43M [00:00<00:00, 21.6MB/s]

Checkpoint: /home/mamem/research/interatomic-bond-analyzer/s128_t64.nequip.ckpt


## 2. Read E3D parameters and load the model

In [4]:
checkpoint = torch.load(
    str(ckpt_path),
    map_location=device,
    weights_only=False,
)

avgn = 1.0
eshft = 0.0
escal = 1.0

model_config = checkpoint["hyper_parameters"]["model"]
inner_model_config = model_config.get("model", {})

from nequip.utils.global_state import set_global_state

if inner_model_config.get("avg_num_neighbors"):
    avgn = inner_model_config["avg_num_neighbors"]

sgij = 1.0 / np.sqrt(2.0 * avgn)

if "per_type_energy_shifts" in inner_model_config:
    shift = inner_model_config["per_type_energy_shifts"]
    if np.any(np.abs(np.asarray(shift)) > 0.0):
        eshft = shift

if "per_type_energy_scales" in inner_model_config:
    scale = inner_model_config["per_type_energy_scales"]
    if np.any(np.abs(np.asarray(scale)) > 0.0):
        escal = scale

print("avgn =", avgn)
print("sgij =", sgij)
print("eshft =", eshft)
print("escal =", escal)

set_global_state(allow_tf32=True)


avgn = 25.664177947572092
sgij = 0.1395793987481322
eshft = 0.0
escal = 0.9731940477594199


In [5]:
from nequip.scripts.compile import load_saved_model, _EAGER_MODEL_KEY, _SOLE_MODEL_KEY
from nequip.nn import graph_model

model = load_saved_model(
    str(ckpt_path),
    _EAGER_MODEL_KEY,
    _SOLE_MODEL_KEY,
)
model = model.to(device)
model.eval()

type_names = model.metadata[graph_model.TYPE_NAMES_KEY].split(" ")
r_max = float(model.metadata[graph_model.R_MAX_KEY])

print("Elements in model:", type_names)
print("r_max (Å):", r_max)
print("Device:", device)


[2026-10-04 13:35:30,117][nequip.model.saved_models.load_utils][INFO] - [rank: 0] Loading model from /home/mamem/research/interatomic-bond-analyzer/s128_t64.nequip.ckpt ...
[2026-10-04 13:35:30,120][nequip.model.saved_models.checkpoint][INFO] - [rank: 0] Loading model from checkpoint file: /home/mamem/research/interatomic-bond-analyzer/s128_t64.nequip.ckpt ...


/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/nequip/model/saved_models/checkpoint.py:69: UserWarning: `torch` versions differ between the checkpoint file (2.7.1) and the current run (2.14.1+cpu) -- `ModelFromCheckpoint` will be built with the current run's versions, but please check that this decision is as intended.
  warnings.warn(
/home/mamem/miniforge3/envs/bond-analyzer/lib/python3.10/site-packages/nequip/model/saved_models/checkpoint.py:69: UserWarning: `e3nn` versions differ between the checkpoint file (0.5.6) and the current run (0.6.0) -- `ModelFromCheckpoint` will be built with the current run's versions, but please check that this decision is as intended.
  warnings.warn(
/home/

[2026-10-04 13:35:31,605][nequip.train.ema][INFO] - [rank: 0] Loading EMA weights for evaluation model.
Elements in model: ['H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne', 'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'I', 'Xe', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu']
r_max (Å): 6.0
Device: cpu


## 3. Helper function for reading an extended XYZ file

In [6]:
from torch.utils.data import DataLoader
from nequip.data import AtomicDataDict
from nequip.data.dataset import ASEDataset
from nequip.data.transforms import NeighborListTransform, ChemicalSpeciesToAtomTypeMapper

def make_dataloader_for_xyz(xyz_path, chemical_symbols, r_max, device):
    transforms = [
        NeighborListTransform(r_max=r_max),
        ChemicalSpeciesToAtomTypeMapper(chemical_symbols),
    ]

    dataset = ASEDataset(str(xyz_path), transforms=transforms)
    dl = DataLoader(
        dataset,
        batch_size=1,
        collate_fn=AtomicDataDict.batched_from_list,
    )

    def move_to_device(batch, device):
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device)
        return batch

    class DeviceDataLoader:
        def __init__(self, dl, device):
            self.dl = dl
            self.device = device

        def __iter__(self):
            for batch in self.dl:
                yield move_to_device(batch, self.device)

        def __len__(self):
            return len(self.dl)

    return DeviceDataLoader(dl, device)


## 4. Tutorial: a small organic molecule from Transition1x

In [7]:
Tx00001_reactant = """9
Properties=species:S:1:pos:R:3:force:R:3 energy_psi4=-38.57683543154053 formula=C2H3N3O energy_psi4_wb97m-d3bj=-8642.88690309325 energy=-52.1430908541206 methods=wb97m-d3bj,def2-tzvppd pbc="F F F"
C       -1.31180026       0.00791080       0.03195078      -0.11373426      -0.01058038       0.00279037
C        0.17396933      -0.03116598       0.02107604      -0.06128770      -0.22221092       0.15906453
N        0.90716467      -0.90183274       0.64560492      -0.32271012       0.22592169      -0.15905621
O        2.17285288      -0.54960924       0.34537693       0.24154701      -0.05762860       0.03983627
N        2.15838307       0.54406936      -0.46705619      -0.27106724       0.13737833      -0.09275633
N        0.96356106       0.86771307      -0.67129698       0.25400821      -0.08778141       0.06193242
H       -1.66526625       0.95144201       0.45677812       0.09198014      -0.16605963      -0.07834992
H       -1.69759361      -0.06712209      -0.98825892       0.08549051       0.01661658       0.17971440
H       -1.70171442      -0.82112755       0.62509011       0.09558837       0.16430347      -0.11304641
"""

xyz_path = WORKDIR / "Tx00001R.xyz"
with open(xyz_path, "w") as f:
    f.write(Tx00001_reactant)

### Visualize the molecule

The numbers shown later refer to the atom indices in this structure.  
At this stage, simply confirm that the molecular structure is displayed.


In [8]:
from ase.io import read

# Read the same XYZ file with ASE so the atom indices and coordinates are explicit.
atoms = read(xyz_path)

with open(xyz_path) as f:
    xyz_data = f.read()

view = py3Dmol.view(width=700, height=450)
view.addModel(xyz_data, "xyz")
view.setStyle({
    "stick": {"radius": 0.15},
    "sphere": {"scale": 0.30},
})

# Show atom labels as Element + zero-based atom ID: C0, N2, ...
for atom_id, (symbol, pos) in enumerate(
    zip(atoms.get_chemical_symbols(), atoms.positions)
):
    view.addLabel(
        f"{symbol}{atom_id}",
        {
            "position": {
                "x": float(pos[0]),
                "y": float(pos[1]),
                "z": float(pos[2]),
            },
            "fontSize": 12,
            "fontColor": "black",
            "backgroundColor": "white",
            "backgroundOpacity": 0.75,
            "borderColor": "gray",
            "borderThickness": 1,
            "inFront": True,
        },
    )

view.zoomTo()
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
dl = make_dataloader_for_xyz(xyz_path, type_names, r_max, device)
print("Input structure is ready.")


Input structure is ready.


## 5. Calculate and print E3D pair energies

For each atomic pair within the model cutoff, the code prints:

- atom indices,
- element pair,
- interatomic distance \(r\),
- E3D pair contribution \(D_{ij}\) in eV.

For the first exercise, do not try to understand the full implementation below. Focus on the **final table** produced by the next cell.


In [10]:
from nequip.nn.pair_potential import ZBL

zbl_module = ZBL(
    type_names,
    type_names,
    units="metal",
    irreps_in={AtomicDataDict.NORM_LENGTH_KEY: "0e"},
).to(device)

def print_edge_energies(out, chemical_symbols, zbl_module=None, cutoff=5.0):
    DKEY = AtomicDataDict

    edge_eng = out[DKEY.EDGE_ENERGY_KEY].squeeze().cpu().detach().numpy()
    edge_idx = out[DKEY.EDGE_INDEX_KEY].squeeze().cpu().detach().numpy().T
    edge_r = out[DKEY.EDGE_LENGTH_KEY].squeeze().cpu().detach().numpy()
    types = out[DKEY.ATOM_TYPE_KEY].squeeze().cpu().detach().numpy()

    edge_eng_zbl = None

    zbl_key = getattr(DKEY, "EDGE_ENERGY_ZBL_KEY", None)
    if zbl_key is not None and zbl_key in out:
        edge_eng_zbl = out[zbl_key].squeeze().cpu().detach().numpy()
    else:
        for try_key in ("edge_energy_zbl", "EDGE_ENERGY_ZBL_KEY", "edge_energy_zbl_key"):
            if try_key in out:
                edge_eng_zbl = out[try_key].squeeze().cpu().detach().numpy()
                break

    if edge_eng_zbl is None and zbl_module is not None:
        try:
            zbl_out = zbl_module._zbl(
                Z=zbl_module.atomic_numbers,
                r=out[DKEY.EDGE_LENGTH_KEY].view(-1),
                atom_types=out[DKEY.ATOM_TYPE_KEY],
                edge_index=out[DKEY.EDGE_INDEX_KEY],
                qqr2exesquare=zbl_module._qqr2exesquare,
            )
            edge_eng_zbl = zbl_out.squeeze().cpu().detach().numpy()

            zbl_out_cutoff = zbl_module.cutoff(
                out[DKEY.NORM_LENGTH_KEY]
            ).to(torch.get_default_dtype())
            zbl_edge_cutoff = zbl_out_cutoff.squeeze().cpu().detach().numpy()

            edge_eng_zbl = edge_eng_zbl * zbl_edge_cutoff
        except Exception as exc:
            print("Warning: ZBL term could not be evaluated:", exc)
            edge_eng_zbl = np.zeros_like(edge_eng)

    if edge_eng_zbl is None:
        edge_eng_zbl = np.zeros_like(edge_eng)

    print("idx_i idx_j  sym_i-sym_j      r (Å)    D_ij (eV)")
    print("-" * 55)

    sumint = 0.0
    sumdij = 0.0
    sumzbl = 0.0
    cnt = 0
    rec = {}

    for e_idx, eng, eng_zbl, r in zip(edge_idx, edge_eng, edge_eng_zbl, edge_r):
        r_val = float(r)
        if r_val > cutoff:
            continue

        i_idx = int(e_idx[0])
        j_idx = int(e_idx[1])

        key = (i_idx, j_idx, round(r_val, 6))
        rev = (j_idx, i_idx, round(r_val, 6))

        if rev in rec:
            other = rec.pop(rev)

            dij_core = (float(eng) + other["eng"]) * sgij * escal
            dij_core = float(np.asarray(dij_core).squeeze())

            zbl_pair = 2.0 * float(eng_zbl)
            dij_total = dij_core + zbl_pair

            sumdij += dij_core
            sumzbl += zbl_pair
            sumint += dij_total
            cnt += 1

            sym_i = chemical_symbols[int(types[i_idx])]
            sym_j = chemical_symbols[int(types[j_idx])]

            print(
                f"{i_idx:3d}-{j_idx:3d}   "
                f"{sym_i:>2s}-{sym_j:<2s}      "
                f"{r_val:7.3f}   {dij_total:10.4f}"
            )
        else:
            rec[key] = {"eng": float(eng), "eng_zbl": float(eng_zbl)}

    print("-" * 55)
    print(f"Number of pairs : {cnt}")
    print(f"Sum D_ij        : {sumint:.6f} eV")
    print(f"Core term sum   : {sumdij:.6f} eV")
    print(f"ZBL term sum    : {sumzbl:.6f} eV")

    if rec:
        print(f"Warning: {len(rec)} directed edges did not find a reverse partner.")


In [11]:
# Keep gradient tracking enabled.
# This NequIP/Allegro model computes forces internally using autograd.
for batch in dl:
    out = model(batch)
    print_edge_energies(
        out,
        chemical_symbols=type_names,
        zbl_module=zbl_module,
        cutoff=r_max,
    )
    break


idx_i idx_j  sym_i-sym_j      r (Å)    D_ij (eV)
-------------------------------------------------------
  1-  0    C-C         1.486      -4.3108
  2-  0    N-C         2.475       0.0477
  2-  1    N-C         1.298      -5.1836
  3-  0    O-C         3.543      -0.0142
  3-  1    O-C         2.090      -0.0035
  3-  2    O-N         1.348      -2.8785
  4-  0    N-C         3.547      -0.0083
  4-  1    N-C         2.123       0.1279
  4-  2    N-N         2.212      -0.1648
  4-  3    N-O         1.362      -2.7981
  5-  0    N-C         2.532       0.0134
  5-  1    N-C         1.382      -4.4081
  5-  2    N-N         2.207       0.1912
  5-  3    N-O         2.122       0.0730
  5-  4    N-N         1.255      -4.7813
  6-  0    H-C         1.093      -4.2496
  6-  1    H-C         2.130      -0.0538
  6-  2    H-N         3.176      -0.0259
  6-  3    H-O         4.123      -0.0113
  6-  4    H-N         3.955      -0.0106
  6-  5    H-N         2.862      -0.0546
  7-  0    H-

## 6. Inspect one selected atomic pair

Use the atom IDs shown in the 3D structure above.

For example,

```python
atom_i = 0
atom_j = 1
```

prints only the information for the pair **atom 0 – atom 1**:

- element symbols,
- interatomic distance,
- E3D pair energy \(D_{ij}\).

The atom IDs are **zero-based**, so the first atom is ID `0`.


In [21]:
def show_pair_energy(out, atom_i, atom_j, chemical_symbols, zbl_module=None):
    """Print distance and E3D D_ij for one selected atom pair."""
    DKEY = AtomicDataDict

    atom_i = int(atom_i)
    atom_j = int(atom_j)

    if atom_i == atom_j:
        raise ValueError("Please choose two different atom IDs.")

    edge_eng = out[DKEY.EDGE_ENERGY_KEY].squeeze().detach().cpu().numpy()
    edge_idx = out[DKEY.EDGE_INDEX_KEY].squeeze().detach().cpu().numpy().T
    edge_r = out[DKEY.EDGE_LENGTH_KEY].squeeze().detach().cpu().numpy()
    types = out[DKEY.ATOM_TYPE_KEY].squeeze().detach().cpu().numpy()

    n_atoms = len(types)
    if not (0 <= atom_i < n_atoms and 0 <= atom_j < n_atoms):
        raise IndexError(
            f"Atom IDs must be between 0 and {n_atoms - 1}. "
            f"Received {atom_i} and {atom_j}."
        )

    # Compute the ZBL edge contribution in the same way as the table above.
    edge_eng_zbl = None

    zbl_key = getattr(DKEY, "EDGE_ENERGY_ZBL_KEY", None)
    if zbl_key is not None and zbl_key in out:
        edge_eng_zbl = out[zbl_key].squeeze().detach().cpu().numpy()
    else:
        for try_key in (
            "edge_energy_zbl",
            "EDGE_ENERGY_ZBL_KEY",
            "edge_energy_zbl_key",
        ):
            if try_key in out:
                edge_eng_zbl = out[try_key].squeeze().detach().cpu().numpy()
                break

    if edge_eng_zbl is None and zbl_module is not None:
        try:
            zbl_out = zbl_module._zbl(
                Z=zbl_module.atomic_numbers,
                r=out[DKEY.EDGE_LENGTH_KEY].view(-1),
                atom_types=out[DKEY.ATOM_TYPE_KEY],
                edge_index=out[DKEY.EDGE_INDEX_KEY],
                qqr2exesquare=zbl_module._qqr2exesquare,
            )
            edge_eng_zbl = zbl_out.squeeze().detach().cpu().numpy()

            zbl_out_cutoff = zbl_module.cutoff(
                out[DKEY.NORM_LENGTH_KEY]
            ).to(torch.get_default_dtype())
            zbl_edge_cutoff = zbl_out_cutoff.squeeze().detach().cpu().numpy()
            edge_eng_zbl = edge_eng_zbl * zbl_edge_cutoff
        except Exception as exc:
            print("Warning: ZBL term could not be evaluated:", exc)
            edge_eng_zbl = np.zeros_like(edge_eng)

    if edge_eng_zbl is None:
        edge_eng_zbl = np.zeros_like(edge_eng)

    # Locate both directed edges: i -> j and j -> i.
    idx_ij = None
    idx_ji = None
    for edge_no, (i, j) in enumerate(edge_idx):
        i = int(i)
        j = int(j)
        if i == atom_i and j == atom_j:
            idx_ij = edge_no
        elif i == atom_j and j == atom_i:
            idx_ji = edge_no

    sym_i = chemical_symbols[int(types[atom_i])]
    sym_j = chemical_symbols[int(types[atom_j])]

    if idx_ij is None or idx_ji is None:
        positions = out[DKEY.POSITIONS_KEY].squeeze().detach().cpu().numpy()
        distance = float(np.linalg.norm(positions[atom_i] - positions[atom_j]))

        print("Selected atomic pair")
        print("--------------------")
        print(f"Atoms    : {sym_i}{atom_i} - {sym_j}{atom_j}")
        print(f"Distance : {distance:.4f} Å")
        print(
            f"D_ij     : not available "
            f"(the pair is outside the model cutoff r_max = {r_max:.3f} Å)"
        )
        return

    # E3D pair energy: combine the two directed edges.
    dij_core = (
        float(edge_eng[idx_ij]) + float(edge_eng[idx_ji])
    ) * sgij * escal
    dij_core = float(np.asarray(dij_core).squeeze())

    zbl_pair = float(edge_eng_zbl[idx_ij]) + float(edge_eng_zbl[idx_ji])
    dij_total = dij_core + zbl_pair
    distance = float(edge_r[idx_ij])

    print("Selected atomic pair")
    print("--------------------")
    print(f"Atoms    : {sym_i}{atom_i} - {sym_j}{atom_j}")
    print(f"Distance : {distance:.4f} Å")
    print(f"D_ij     : {dij_total:.6f} eV")


In [22]:
# Change only these two atom IDs.
atom_i = 0
atom_j = 1

show_pair_energy(
    out,
    atom_i,
    atom_j,
    chemical_symbols=type_names,
    zbl_module=zbl_module,
)


Selected atomic pair
--------------------
Atoms    : C0 - C1
Distance : 1.4863 Å
D_ij     : -4.310760 eV


## 7. First exercise

For now, answer only these questions.

1. Looking at the molecular structure, which atom pairs appear to be covalently bonded?
2. Find those pairs in the \(D_{ij}\) output.
3. Compare their distances and \(D_{ij}\) values.
4. Are there short-distance pairs that do **not** look like ordinary covalent bonds?

For the selected-pair exercise, change only `atom_i` and `atom_j`. You do not need to modify the analysis functions.

### Next step

After this tutorial works, we can replace the molecule and ask a research question such as:

- Does E3D behave similarly for bonds containing elements other than C/H/N/O?
- What happens for hydrogen-bonded molecular pairs?
- What happens for ionic systems?

The important first step is simply to make the calculation run and learn how to read the output.
